## Understanding Precision, Recall, and F1

Think of it like this:

- **Precision** = “When the model says this is a business, how often is it correct?”
- **Recall** = “Out of all real business names present, how many did the model actually find?”
- **F1** = “Overall balance between not making wrong guesses and not missing businesses.”

---

## Result Interpretation

| Model Result | What it means |
|---|---|
| **spaCy Precision = 1.0** | Everything spaCy identified as a business was correct. It made **no wrong guesses**. |
| **spaCy Recall = 0.44** | It found only about **44% of the actual businesses**, so it missed more than half. |
| **spaCy F1 = 0.61** | Its overall performance is lower because, although its predictions were correct, it **missed many businesses**. |
| **GLiNER Precision = 1.0** | Everything GLiNER extracted was correct. |
| **GLiNER Recall = 1.0** | It found **every business mention** in the test data. |
| **GLiNER F1 = 1.0** | It achieved a perfect result on this particular test set. |

---

## Simple Example

Suppose there are **9 actual business names** in the test data.

### spaCy

```text
Actual businesses: 9

spaCy found: 4
Correct: 4
Wrong guesses: 0
Missed: 5

In [9]:
import spacy
from gliner import GLiNER

In [14]:
# spaCy
spacy_ner = spacy.load("D:/Sculptsoft/business-mention-resolution-platform/.venv/Lib/site-packages/en_core_web_sm/en_core_web_sm-3.8.0")

# GLiNER
gliner_model = GLiNER.from_pretrained(
    "gliner-community/gliner_small-v2.5"
)

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 270.48it/s]


# SpaCy NER

In [19]:
text = "John wants to buy Apple phone in india on january 5 2026, after eating pizza at Domino's near Iskon cross road in Ahmedabad"

In [20]:
doc = spacy_ner(text)

for ent in doc.ents:
    print(ent.text, "->", ent.label_)

John -> PERSON
Apple -> ORG
india -> GPE
january 5 2026 -> DATE
Domino -> PERSON
Iskon -> NORP
Ahmedabad -> GPE


# GLiNER

In [24]:
labels = [
    "business",
    "restaurant",
    "store",
    "hotel",
    "cafe",
    "name",
    "address",
    "city",
    "food"
]

entities = gliner_model.predict_entities(
    text,
    labels,
    threshold=0.45
)

entities

[{'start': 0,
  'end': 4,
  'text': 'John',
  'label': 'name',
  'score': 0.9707052707672119},
 {'start': 18,
  'end': 23,
  'text': 'Apple',
  'label': 'business',
  'score': 0.5099247097969055},
 {'start': 71,
  'end': 76,
  'text': 'pizza',
  'label': 'food',
  'score': 0.9044143557548523},
 {'start': 80,
  'end': 88,
  'text': "Domino's",
  'label': 'store',
  'score': 0.5135442614555359},
 {'start': 94,
  'end': 110,
  'text': 'Iskon cross road',
  'label': 'address',
  'score': 0.8927795886993408},
 {'start': 114,
  'end': 123,
  'text': 'Ahmedabad',
  'label': 'city',
  'score': 0.9924561977386475}]

In [25]:
for entity in entities:
    print(
        entity["text"],
        "->",
        entity["label"],
        "confidence:",
        round(entity["score"], 3)
    )

John -> name confidence: 0.971
Apple -> business confidence: 0.51
pizza -> food confidence: 0.904
Domino's -> store confidence: 0.514
Iskon cross road -> address confidence: 0.893
Ahmedabad -> city confidence: 0.992


### Test Model

In [26]:
test_data = [
    {
        "text": "We ordered pizza from Domino's yesterday.",
        "expected": ["Domino's"]
    },
    {
        "text": "I had coffee at Starbucks before work.",
        "expected": ["Starbucks"]
    },
    {
        "text": "The Hilton near downtown Chicago was excellent.",
        "expected": ["Hilton"]
    },
    {
        "text": "Dinner at Joe's Pizza was amazing.",
        "expected": ["Joe's Pizza"]
    },
    {
        "text": "I bought groceries from Whole Foods Market.",
        "expected": ["Whole Foods Market"]
    },
    {
        "text": "John met Sarah near Central Park.",
        "expected": []
    },
    {
        "text": "The coffee at Blue Bottle Coffee was fantastic.",
        "expected": ["Blue Bottle Coffee"]
    },
    {
        "text": "I visited Target on Saturday.",
        "expected": ["Target"]
    },
    {
        "text": "We stayed at Marriott during our trip.",
        "expected": ["Marriott"]
    },
    {
        "text": "The Cheesecake Factory was crowded tonight.",
        "expected": ["The Cheesecake Factory"]
    }
]

In [27]:
def extract_spacy(text):
    doc = spacy_ner(text)

    return [
        ent.text
        for ent in doc.ents
        if ent.label_ in ["ORG", "FAC"]
    ]

In [28]:
def extract_gliner(text):
    entities = gliner_model.predict_entities(
        text,
        [
            "business",
            "restaurant",
            "store",
            "hotel",
            "cafe"
        ],
        threshold=0.45
    )

    return [entity["text"] for entity in entities]

In [29]:
def evaluate_model(test_data, extractor):

    TP = 0
    FP = 0
    FN = 0

    for item in test_data:

        expected = set(
            x.lower().strip()
            for x in item["expected"]
        )

        predicted = set(
            x.lower().strip()
            for x in extractor(item["text"])
        )

        TP += len(expected & predicted)
        FP += len(predicted - expected)
        FN += len(expected - predicted)

    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall
        else 0
    )

    return {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [30]:
spacy_results = evaluate_model(
    test_data,
    extract_spacy
)

spacy_results

{'TP': 4,
 'FP': 0,
 'FN': 5,
 'Precision': 1.0,
 'Recall': 0.4444444444444444,
 'F1': 0.6153846153846153}

In [31]:
gliner_results = evaluate_model(
    test_data,
    extract_gliner
)

gliner_results

{'TP': 9, 'FP': 0, 'FN': 0, 'Precision': 1.0, 'Recall': 1.0, 'F1': 1.0}

In [32]:
import pandas as pd

comparison = pd.DataFrame([
    {
        "Model": "spaCy en_core_web_sm",
        "Precision": spacy_results["Precision"],
        "Recall": spacy_results["Recall"],
        "F1": spacy_results["F1"]
    },
    {
        "Model": "GLiNER Small v2.5",
        "Precision": gliner_results["Precision"],
        "Recall": gliner_results["Recall"],
        "F1": gliner_results["F1"]
    }
])

comparison

,Model,Precision,Recall,F1
0,spaCy en_core_web_sm,1.0,0.444444,0.615385
1,GLiNER Small v2.5,1.0,1.000000,1.000000


In [33]:
comparison[["Precision", "Recall", "F1"]] *= 100

comparison.round(2)

,Model,Precision,Recall,F1
0,spaCy en_core_web_sm,100.0,44.44,61.54
1,GLiNER Small v2.5,100.0,100.00,100.00
